In [31]:
import pandas as pd

# Dataset Desmatamento
# Carrega o dataset do desmatamento do bioma amazônia
df1 = pd.read_csv('dataset_desmatamento.csv', sep=';', decimal=',')

# Verifica a existência de dados faltantes
print("\n--- Quantidade de Dados Faltantes (Nulos) por Coluna ---")
print(df1.isnull().sum())

print("\n--- Tipos de Dados de Cada Coluna ---")
print(df1.dtypes)


--- Quantidade de Dados Faltantes (Nulos) por Coluna ---
year            0
areakm          0
municipality    0
geocode_ibge    0
state           0
dtype: int64

--- Tipos de Dados de Cada Coluna ---
year              int64
areakm          float64
municipality     object
geocode_ibge      int64
state            object
dtype: object


In [32]:

# Renomeia as colunas
df1 = df1.rename(columns={
    'year': 'ano',
    'areakm': 'area_desmatada_km2',
    'municipality': 'municipio',
    'geocode_ibge': 'codigo_ibge',
    'state': 'estado'
})

# Padronizar textos: Deixar municípios e estados em MAIÚSCULO
df1['municipio'] = df1['municipio'].str.upper()
df1['estado'] = df1['estado'].str.upper()

# Remover acentos dos municípios (Ex: 'ACRELÂNDIA' vira 'ACRELANDIA')
df1['municipio'] = df1['municipio'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')

# Preparar dados: Arredondar a área desmatada para 2 casas decimais
df1['area_desmatada_km2'] = df1['area_desmatada_km2'].round(2)

# Exibir a tabela limpa, padronizada e pronta para análise!
print("Tabela após Limpeza e Padronização:")
display(df1.head())

Tabela após Limpeza e Padronização:


,ano,area_desmatada_km2,municipio,codigo_ibge,estado
0,2007,1093.83,ACRELANDIA,1200013,ACRE
1,2008,23.67,ACRELANDIA,1200013,ACRE
2,2009,12.94,ACRELANDIA,1200013,ACRE
3,2010,11.33,ACRELANDIA,1200013,ACRE
4,2011,15.94,ACRELANDIA,1200013,ACRE


In [33]:
# Dataset Gado

# Carrega o dataset Gado pulando as 4 primeiras linhas (que são apenas textos de cabeçalho do site)
df2 = pd.read_csv('dataset_gado.csv', sep=';', skiprows=4)

# "Derrete" a tabela: transforma as várias colunas espalhadas (Bovino, Bovino.1) em linhas, criando uma coluna para o ano e outra para a quantidade
df2_limpo = df2.melt(id_vars=['Cód.', 'Município'],
                     var_name='ano_original',
                     value_name='rebanho_bovino')

# Muda o nome da coluna 'Cód.' para 'codigo_ibge' para ficar idêntico ao dataset de desmatamento
df2_limpo = df2_limpo.rename(columns={'Cód.': 'codigo_ibge'})

# Pega a coluna com os nomes antigos ('Bovino.1'), extrai só o número ('1') e preenche com zero se não tiver número nenhum
df2_limpo['incremento_ano'] = df2_limpo['ano_original'].str.extract(r'(\d+)').fillna(0).astype(int)

# Cria a coluna final de 'ano' somando o ano inicial da pesquisa (1974) com o número extraído na linha de cima
df2_limpo['ano'] = 1974 + df2_limpo['incremento_ano']

# Joga fora as colunas temporárias e o nome do município, deixando a tabela mais limpa e leve
df2_limpo = df2_limpo.drop(columns=['ano_original', 'incremento_ano', 'Município'])

# Força o código do IBGE a ser interpretado como texto (string) para não dar erro na hora de juntar com a outra tabela
df2_limpo['codigo_ibge'] = df2_limpo['codigo_ibge'].astype(str)

# Converte o rebanho para número. Se o IBGE tiver colocado algum traço ou texto (erro), ele transforma em zero
df2_limpo['rebanho_bovino'] = pd.to_numeric(df2_limpo['rebanho_bovino'], errors='coerce').fillna(0)

print("--- TABELA DE GADO COM OS ANOS CORRETOS ---")

display(df2_limpo.head())

--- TABELA DE GADO COM OS ANOS CORRETOS ---


,codigo_ibge,rebanho_bovino,ano
0,1100015,0.0,1974
1,1100023,0.0,1974
2,1100031,0.0,1974
3,1100049,0.0,1974
4,1100056,0.0,1974


In [34]:
# Converte o código da cidade no dataset de desmatamento (df1) para texto (string)
df1['codigo_ibge'] = df1['codigo_ibge'].astype(str)

# Faz a mesma coisa no dataset do gado (df2_limpo) para garantir que as duas tabelas falem a mesma língua
df2_limpo['codigo_ibge'] = df2_limpo['codigo_ibge'].astype(str)

# Transforma a coluna de ano no dataset de desmatamento em número inteiro (sem casas decimais)
df1['ano'] = df1['ano'].astype(int)

# Transforma a coluna de ano no dataset de gado em número inteiro também
df2_limpo['ano'] = df2_limpo['ano'].astype(int)

# Junta as duas tabelas (df1 e df2_limpo) usando o código da cidade e o ano como chaves.
df_final = pd.merge(df1, df2_limpo, on=['codigo_ibge', 'ano'], how='inner')

print("\n--- O DATASET FINAL UNIFICADO! ---")

display(df_final.head())


--- O DATASET FINAL UNIFICADO! ---


,ano,area_desmatada_km2,municipio,codigo_ibge,estado,rebanho_bovino
0,2007,1093.83,ACRELANDIA,1200013,ACRE,168142.0
1,2008,23.67,ACRELANDIA,1200013,ACRE,185359.0
2,2009,12.94,ACRELANDIA,1200013,ACRE,185359.0
3,2010,11.33,ACRELANDIA,1200013,ACRE,188756.0
4,2011,15.94,ACRELANDIA,1200013,ACRE,173174.0
